# Viral Content Predictor using LlamaIndex Agents

This agent uses LlamaIndex's agent framework to create a sophisticated content analysis and prediction system that:

- **Analyzes trending content** across multiple social media platforms
- **Predicts virality scores** using machine learning and engagement patterns
- **Generates content variations** optimized for different platforms
- **Performs A/B testing** and real-time optimization
- **Tracks performance metrics** and learns from engagement data

Unlike Haystack's pipeline approach, LlamaIndex uses tool-calling agents with advanced query engines and vector-based content understanding for nuanced social media analysis.

In [ ]:
# Install required packages
%pip install --upgrade --quiet llama-index
%pip install --upgrade --quiet llama-index-agent-openai
%pip install --upgrade --quiet llama-index-llms-openai
%pip install --upgrade --quiet llama-index-embeddings-openai
%pip install --upgrade --quiet llama-index-tools-requests
%pip install --upgrade --quiet llama-index-vector-stores-chroma
%pip install --upgrade --quiet requests
%pip install --upgrade --quiet tweepy
%pip install --upgrade --quiet praw  # Reddit API
%pip install --upgrade --quiet beautifulsoup4
%pip install --upgrade --quiet pandas
%pip install --upgrade --quiet numpy
%pip install --upgrade --quiet scikit-learn
%pip install --upgrade --quiet textstat
%pip install --upgrade --quiet vaderSentiment
%pip install --upgrade --quiet chromadb

In [ ]:
import os
import json
import time
import pandas as pd
import numpy as np
from pathlib import Path
from typing import Dict, List, Optional, Any, Tuple
from datetime import datetime, timedelta
from dataclasses import dataclass, asdict
import requests
import re
import asyncio
from collections import defaultdict
import math

# Text analysis libraries
import textstat
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

# Machine learning
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

# LlamaIndex imports
from llama_index.core import VectorStoreIndex, Document, Settings
from llama_index.core.agent import ReActAgent
from llama_index.core.tools import BaseTool, FunctionTool
from llama_index.core.llms import ChatMessage
from llama_index.llms.openai import OpenAI
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.vector_stores.chroma import ChromaVectorStore
from llama_index.core import StorageContext
import chromadb

# Import helper for environment variables
import sys
sys.path.append('..')
from helpers import set_environment_variables

In [ ]:
# Set up environment variables
set_environment_variables()

# Configure LlamaIndex
Settings.llm = OpenAI(
    model="gpt-3.5-turbo",
    api_key=os.getenv("OPENAI_API_KEY"),
    temperature=0.1
)

Settings.embed_model = OpenAIEmbedding(
    api_key=os.getenv("OPENAI_API_KEY")
)

# Initialize ChromaDB for vector storage
chroma_client = chromadb.EphemeralClient()
chroma_collection = chroma_client.create_collection("viral_content")
vector_store = ChromaVectorStore(chroma_collection=chroma_collection)
storage_context = StorageContext.from_defaults(vector_store=vector_store)

print("LlamaIndex components initialized successfully!")

## Content Analysis and Prediction Models

Let's create models for analyzing and predicting viral content:

In [ ]:
@dataclass
class ContentPost:
    """Represents a social media post"""
    id: str
    platform: str
    content: str
    author: str
    timestamp: datetime
    engagement: Dict[str, int]  # likes, shares, comments, views
    hashtags: List[str]
    mentions: List[str]
    media_type: str  # text, image, video, link
    url: Optional[str] = None
    metadata: Dict[str, Any] = None
    
    def __post_init__(self):
        if self.metadata is None:
            self.metadata = {}

@dataclass
class ViralityPrediction:
    """Represents a virality prediction"""
    content: str
    platform: str
    virality_score: float  # 0-100
    engagement_prediction: Dict[str, int]
    confidence: float
    factors: Dict[str, float]  # Contributing factors
    recommendations: List[str]
    optimal_timing: datetime
    hashtag_suggestions: List[str]

class ContentAnalyzer:
    """Analyzes content for viral potential"""
    
    def __init__(self):
        self.sentiment_analyzer = SentimentIntensityAnalyzer()
        self.tfidf = TfidfVectorizer(max_features=1000, stop_words='english')
        self.scaler = StandardScaler()
        self.model = None
        self.is_trained = False
        
        # Viral content patterns (learned from data)
        self.viral_patterns = {
            'emotional_triggers': ['amazing', 'shocking', 'unbelievable', 'incredible', 'mind-blowing'],
            'action_words': ['discover', 'reveal', 'expose', 'unlock', 'transform'],
            'time_indicators': ['now', 'today', 'just', 'breaking', 'urgent'],
            'social_proof': ['everyone', 'millions', 'trending', 'viral', 'popular']
        }
    
    def analyze_content_features(self, content: str) -> Dict[str, float]:
        """Extract features from content that correlate with virality"""
        features = {}
        
        # Basic metrics
        features['length'] = len(content)
        features['word_count'] = len(content.split())
        features['char_count'] = len(content)
        features['sentence_count'] = content.count('.') + content.count('!') + content.count('?')
        
        # Readability scores
        features['flesch_reading_ease'] = textstat.flesch_reading_ease(content)
        features['flesch_kincaid_grade'] = textstat.flesch_kincaid_grade(content)
        features['smog_index'] = textstat.smog_index(content)
        
        # Sentiment analysis
        sentiment_scores = self.sentiment_analyzer.polarity_scores(content)
        features['sentiment_positive'] = sentiment_scores['pos']
        features['sentiment_negative'] = sentiment_scores['neg']
        features['sentiment_neutral'] = sentiment_scores['neu']
        features['sentiment_compound'] = sentiment_scores['compound']
        
        # Emotional intensity
        features['exclamation_count'] = content.count('!')
        features['question_count'] = content.count('?')
        features['caps_ratio'] = sum(1 for c in content if c.isupper()) / len(content) if content else 0
        
        # Social media specific
        features['hashtag_count'] = content.count('#')
        features['mention_count'] = content.count('@')
        features['emoji_count'] = len([c for c in content if ord(c) > 127])
        features['url_count'] = len(re.findall(r'http[s]?://(?:[a-zA-Z]|[0-9]|[$-_@.&+]|[!*\(\),]|(?:%[0-9a-fA-F][0-9a-fA-F]))+', content))
        
        # Viral pattern matching
        content_lower = content.lower()
        for pattern_type, patterns in self.viral_patterns.items():
            count = sum(1 for pattern in patterns if pattern in content_lower)
            features[f'{pattern_type}_count'] = count
        
        # Timing and structure
        features['has_numbers'] = int(bool(re.search(r'\d+', content)))
        features['starts_with_question'] = int(content.strip().startswith(('What', 'How', 'Why', 'When', 'Where')))
        features['has_call_to_action'] = int(any(cta in content_lower for cta in ['share', 'like', 'comment', 'retweet', 'follow']))
        
        return features
    
    def predict_virality(self, content: str, platform: str = 'twitter') -> ViralityPrediction:
        """Predict virality score for content"""
        features = self.analyze_content_features(content)
        
        # Base score calculation (simplified model)
        base_score = 30  # Average content baseline
        
        # Adjust based on features
        if features['sentiment_positive'] > 0.3:
            base_score += 10
        if features['emotional_triggers_count'] > 0:
            base_score += 15
        if features['has_call_to_action']:
            base_score += 10
        if 50 < features['word_count'] < 150:  # Optimal length
            base_score += 8
        if features['hashtag_count'] > 0:
            base_score += min(features['hashtag_count'] * 3, 12)
        
        # Platform-specific adjustments
        if platform == 'twitter':
            if features['word_count'] < 280:  # Twitter character limit consideration
                base_score += 5
        elif platform == 'linkedin':
            if 100 < features['word_count'] < 300:  # LinkedIn optimal length
                base_score += 8
        elif platform == 'tiktok':
            if features['has_numbers']:  # TikTok likes numbered content
                base_score += 7
        
        # Cap the score
        virality_score = min(base_score, 100)
        
        # Generate predictions
        engagement_multiplier = virality_score / 50  # Base multiplier
        
        engagement_prediction = {
            'likes': int(100 * engagement_multiplier),
            'shares': int(20 * engagement_multiplier),
            'comments': int(15 * engagement_multiplier),
            'views': int(1000 * engagement_multiplier)
        }
        
        # Generate recommendations
        recommendations = []
        if features['sentiment_compound'] < 0.2:
            recommendations.append("Consider adding more positive language")
        if features['hashtag_count'] == 0:
            recommendations.append("Add 2-3 relevant hashtags")
        if not features['has_call_to_action']:
            recommendations.append("Include a call-to-action (share, like, comment)")
        if features['word_count'] > 200:
            recommendations.append("Consider shortening the content for better engagement")
        
        # Suggest hashtags based on content
        hashtag_suggestions = self._suggest_hashtags(content)
        
        # Optimal timing (simplified)
        optimal_timing = datetime.now() + timedelta(hours=2)
        
        return ViralityPrediction(
            content=content,
            platform=platform,
            virality_score=virality_score,
            engagement_prediction=engagement_prediction,
            confidence=0.75,  # Model confidence
            factors=features,
            recommendations=recommendations,
            optimal_timing=optimal_timing,
            hashtag_suggestions=hashtag_suggestions
        )
    
    def _suggest_hashtags(self, content: str) -> List[str]:
        """Suggest relevant hashtags based on content"""
        # Simple keyword-based hashtag suggestion
        hashtag_map = {
            'ai': ['#AI', '#ArtificialIntelligence', '#MachineLearning'],
            'tech': ['#Tech', '#Technology', '#Innovation'],
            'business': ['#Business', '#Entrepreneurship', '#Startup'],
            'success': ['#Success', '#Motivation', '#Growth'],
            'tips': ['#Tips', '#Advice', '#HowTo'],
            'news': ['#News', '#Breaking', '#Update']
        }
        
        content_lower = content.lower()
        suggested = []
        
        for keyword, hashtags in hashtag_map.items():
            if keyword in content_lower:
                suggested.extend(hashtags[:2])  # Add first 2 hashtags
        
        return suggested[:5]  # Return max 5 suggestions
    
    def train_model(self, posts: List[ContentPost]):
        """Train the virality prediction model on historical data"""
        if len(posts) < 10:
            print("Not enough data to train model")
            return
        
        # Extract features and targets
        X = []
        y = []
        
        for post in posts:
            features = self.analyze_content_features(post.content)
            feature_vector = list(features.values())
            X.append(feature_vector)
            
            # Calculate engagement score as target
            total_engagement = sum(post.engagement.values())
            y.append(total_engagement)
        
        X = np.array(X)
        y = np.array(y)
        
        # Scale features
        X_scaled = self.scaler.fit_transform(X)
        
        # Train model
        self.model = LinearRegression()
        self.model.fit(X_scaled, y)
        self.is_trained = True
        
        print(f"Model trained on {len(posts)} posts")

# Initialize analyzer
analyzer = ContentAnalyzer()
print("Content analyzer initialized!")

## Social Media Data Collection

Let's create tools for collecting trending content data:

In [ ]:
class SocialMediaCollector:
    """Collects trending content from various social media platforms"""
    
    def __init__(self):
        self.platforms = ['twitter', 'reddit', 'linkedin', 'hackernews']
        self.collected_data = []
    
    def collect_trending_content(self, platform: str, limit: int = 50) -> List[ContentPost]:
        """Collect trending content from a specific platform"""
        if platform == 'hackernews':
            return self._collect_hackernews_trending(limit)
        elif platform == 'reddit':
            return self._collect_reddit_trending(limit)
        elif platform == 'mock_data':
            return self._generate_mock_data(limit)
        else:
            print(f"Platform {platform} not implemented yet")
            return []
    
    def _collect_hackernews_trending(self, limit: int) -> List[ContentPost]:
        """Collect trending posts from Hacker News"""
        posts = []
        
        try:
            # Get top stories from HN
            response = requests.get('https://hacker-news.firebaseio.com/v0/topstories.json')
            story_ids = response.json()[:limit]
            
            for story_id in story_ids:
                try:
                    story_response = requests.get(f'https://hacker-news.firebaseio.com/v0/item/{story_id}.json')
                    story_data = story_response.json()
                    
                    if story_data and story_data.get('type') == 'story':
                        post = ContentPost(
                            id=str(story_data['id']),
                            platform='hackernews',
                            content=story_data.get('title', ''),
                            author=story_data.get('by', 'unknown'),
                            timestamp=datetime.fromtimestamp(story_data.get('time', 0)),
                            engagement={
                                'score': story_data.get('score', 0),
                                'comments': story_data.get('descendants', 0),
                                'likes': 0,
                                'shares': 0
                            },
                            hashtags=[],
                            mentions=[],
                            media_type='link' if story_data.get('url') else 'text',
                            url=story_data.get('url'),
                            metadata={'hn_id': story_id}
                        )
                        posts.append(post)
                        
                except Exception as e:
                    continue
            
            print(f"Collected {len(posts)} posts from Hacker News")
            
        except Exception as e:
            print(f"Error collecting from Hacker News: {e}")
        
        return posts
    
    def _collect_reddit_trending(self, limit: int) -> List[ContentPost]:
        """Collect trending posts from Reddit (using public API)"""
        posts = []
        
        try:
            # Get hot posts from r/all
            headers = {'User-Agent': 'ViralContentPredictor/1.0'}
            response = requests.get('https://www.reddit.com/r/all/hot.json?limit=' + str(limit), headers=headers)
            data = response.json()
            
            for item in data['data']['children']:
                post_data = item['data']
                
                post = ContentPost(
                    id=post_data['id'],
                    platform='reddit',
                    content=post_data.get('title', '') + ' ' + post_data.get('selftext', ''),
                    author=post_data.get('author', 'unknown'),
                    timestamp=datetime.fromtimestamp(post_data.get('created_utc', 0)),
                    engagement={
                        'score': post_data.get('score', 0),
                        'comments': post_data.get('num_comments', 0),
                        'likes': post_data.get('ups', 0),
                        'shares': 0
                    },
                    hashtags=[],
                    mentions=[],
                    media_type='image' if post_data.get('url', '').endswith(('.jpg', '.png', '.gif')) else 'text',
                    url=post_data.get('url'),
                    metadata={
                        'subreddit': post_data.get('subreddit', ''),
                        'reddit_id': post_data['id']
                    }
                )
                posts.append(post)
            
            print(f"Collected {len(posts)} posts from Reddit")
            
        except Exception as e:
            print(f"Error collecting from Reddit: {e}")
        
        return posts
    
    def _generate_mock_data(self, limit: int) -> List[ContentPost]:
        """Generate mock social media data for testing"""
        mock_posts = [
            {
                'content': 'Breaking: New AI model achieves human-level performance on complex reasoning tasks! 🤖 #AI #MachineLearning',
                'engagement': {'likes': 1500, 'shares': 300, 'comments': 150, 'views': 15000},
                'platform': 'twitter'
            },
            {
                'content': 'Just discovered this amazing productivity hack that saved me 3 hours every day. Here\'s how you can do it too...',
                'engagement': {'likes': 850, 'shares': 120, 'comments': 80, 'views': 8500},
                'platform': 'linkedin'
            },
            {
                'content': 'This simple morning routine changed my life completely. Try it for 30 days and see the results! ☀️',
                'engagement': {'likes': 2300, 'shares': 450, 'comments': 200, 'views': 23000},
                'platform': 'tiktok'
            },
            {
                'content': 'Scientists discover new species of deep-sea creature that glows in the dark 🌊✨',
                'engagement': {'likes': 980, 'shares': 180, 'comments': 90, 'views': 9800},
                'platform': 'twitter'
            },
            {
                'content': 'The future of remote work: 5 trends that will shape 2024 and beyond',
                'engagement': {'likes': 650, 'shares': 95, 'comments': 55, 'views': 6500},
                'platform': 'linkedin'
            }
        ]
        
        posts = []
        for i, mock in enumerate(mock_posts[:limit]):
            post = ContentPost(
                id=f"mock_{i}",
                platform=mock['platform'],
                content=mock['content'],
                author=f"user_{i}",
                timestamp=datetime.now() - timedelta(hours=i),
                engagement=mock['engagement'],
                hashtags=re.findall(r'#\w+', mock['content']),
                mentions=re.findall(r'@\w+', mock['content']),
                media_type='text',
                metadata={'mock_data': True}
            )
            posts.append(post)
        
        return posts
    
    def analyze_trending_patterns(self, posts: List[ContentPost]) -> Dict[str, Any]:
        """Analyze patterns in trending content"""
        if not posts:
            return {}
        
        # Platform distribution
        platform_counts = defaultdict(int)
        total_engagement = defaultdict(int)
        hashtag_frequency = defaultdict(int)
        
        for post in posts:
            platform_counts[post.platform] += 1
            total_engagement[post.platform] += sum(post.engagement.values())
            
            for hashtag in post.hashtags:
                hashtag_frequency[hashtag] += 1
        
        # Calculate average engagement per platform
        avg_engagement = {}
        for platform in platform_counts:
            avg_engagement[platform] = total_engagement[platform] / platform_counts[platform]
        
        # Top hashtags
        top_hashtags = sorted(hashtag_frequency.items(), key=lambda x: x[1], reverse=True)[:10]
        
        return {
            'total_posts': len(posts),
            'platform_distribution': dict(platform_counts),
            'average_engagement_by_platform': avg_engagement,
            'top_hashtags': top_hashtags,
            'analysis_timestamp': datetime.now().isoformat()
        }

# Initialize collector
collector = SocialMediaCollector()
print("Social media collector initialized!")

## LlamaIndex Agent Tools

Now let's create LlamaIndex tools for the viral content predictor:

In [ ]:
# Create LlamaIndex tools for the agent

def analyze_content_virality(content: str, platform: str = "twitter") -> str:
    """Analyze content and predict its viral potential.
    
    Args:
        content: The text content to analyze
        platform: Target social media platform (twitter, linkedin, tiktok, etc.)
    
    Returns:
        JSON string with virality prediction results
    """
    try:
        prediction = analyzer.predict_virality(content, platform)
        
        result = {
            'virality_score': prediction.virality_score,
            'confidence': prediction.confidence,
            'platform': prediction.platform,
            'engagement_prediction': prediction.engagement_prediction,
            'recommendations': prediction.recommendations,
            'hashtag_suggestions': prediction.hashtag_suggestions,
            'optimal_timing': prediction.optimal_timing.isoformat()
        }
        
        return json.dumps(result, indent=2)
    
    except Exception as e:
        return json.dumps({'error': str(e)})

def collect_trending_content(platform: str = "hackernews", limit: int = 20) -> str:
    """Collect trending content from social media platforms.
    
    Args:
        platform: Platform to collect from (hackernews, reddit, mock_data)
        limit: Number of posts to collect
    
    Returns:
        JSON string with collected posts summary
    """
    try:
        posts = collector.collect_trending_content(platform, limit)
        
        if not posts:
            return json.dumps({'error': 'No posts collected', 'posts_count': 0})
        
        # Analyze the collected content
        patterns = collector.analyze_trending_patterns(posts)
        
        # Store posts in vector database for future analysis
        documents = []
        for post in posts:
            doc = Document(
                text=post.content,
                metadata={
                    'platform': post.platform,
                    'author': post.author,
                    'engagement': post.engagement,
                    'timestamp': post.timestamp.isoformat()
                }
            )
            documents.append(doc)
        
        # Create or update index
        global viral_content_index
        viral_content_index = VectorStoreIndex.from_documents(
            documents, 
            storage_context=storage_context
        )
        
        result = {
            'posts_collected': len(posts),
            'platform': platform,
            'patterns': patterns,
            'sample_posts': [{
                'content': post.content[:100] + '...' if len(post.content) > 100 else post.content,
                'engagement': post.engagement,
                'platform': post.platform
            } for post in posts[:5]]  # Show first 5 posts
        }
        
        return json.dumps(result, indent=2)
    
    except Exception as e:
        return json.dumps({'error': str(e)})

def generate_content_variations(original_content: str, target_platforms: str = "twitter,linkedin") -> str:
    """Generate content variations optimized for different platforms.
    
    Args:
        original_content: The original content to adapt
        target_platforms: Comma-separated list of target platforms
    
    Returns:
        JSON string with content variations
    """
    try:
        platforms = [p.strip() for p in target_platforms.split(',')]
        variations = {}
        
        for platform in platforms:
            if platform == 'twitter':
                # Twitter optimization: shorter, hashtags, emojis
                variation = original_content[:200]  # Twitter length limit consideration
                if not any(hashtag in variation for hashtag in ['#']):
                    variation += " #trending"
                if not any(emoji in variation for emoji in ['🔥', '⚡', '🚀']):
                    variation += " 🔥"
                
            elif platform == 'linkedin':
                # LinkedIn optimization: professional tone, longer form
                variation = original_content
                if not variation.endswith('.'):
                    variation += "."
                variation += "\n\nWhat are your thoughts on this?"
                
            elif platform == 'tiktok':
                # TikTok optimization: casual, trending phrases
                variation = "POV: " + original_content.lower()
                variation += " ✨ #fyp #viral"
                
            else:
                variation = original_content
            
            # Predict virality for each variation
            prediction = analyzer.predict_virality(variation, platform)
            
            variations[platform] = {
                'content': variation,
                'virality_score': prediction.virality_score,
                'recommendations': prediction.recommendations[:3]  # Top 3 recommendations
            }
        
        return json.dumps(variations, indent=2)
    
    except Exception as e:
        return json.dumps({'error': str(e)})

def search_viral_patterns(query: str, top_k: int = 5) -> str:
    """Search for viral content patterns in the collected data.
    
    Args:
        query: Search query for finding similar viral content
        top_k: Number of similar posts to return
    
    Returns:
        JSON string with similar viral content
    """
    try:
        if 'viral_content_index' not in globals():
            return json.dumps({'error': 'No viral content data available. Please collect trending content first.'})
        
        # Query the vector store
        query_engine = viral_content_index.as_query_engine(similarity_top_k=top_k)
        response = query_engine.query(query)
        
        # Extract similar posts
        similar_posts = []
        for node in response.source_nodes:
            similar_posts.append({
                'content': node.text[:150] + '...' if len(node.text) > 150 else node.text,
                'score': node.score,
                'metadata': node.metadata
            })
        
        result = {
            'query': query,
            'response': str(response),
            'similar_posts': similar_posts
        }
        
        return json.dumps(result, indent=2)
    
    except Exception as e:
        return json.dumps({'error': str(e)})

def optimize_posting_time(content: str, platform: str = "twitter", timezone: str = "UTC") -> str:
    """Suggest optimal posting times for maximum engagement.
    
    Args:
        content: Content to optimize timing for
        platform: Target social media platform
        timezone: Target timezone
    
    Returns:
        JSON string with optimal posting recommendations
    """
    try:
        # Platform-specific optimal times (simplified)
        optimal_times = {
            'twitter': {
                'weekdays': ['09:00', '12:00', '15:00', '18:00'],
                'weekends': ['10:00', '14:00', '16:00']
            },
            'linkedin': {
                'weekdays': ['08:00', '12:00', '17:00'],
                'weekends': ['10:00', '14:00']
            },
            'tiktok': {
                'weekdays': ['18:00', '19:00', '20:00'],
                'weekends': ['11:00', '15:00', '19:00']
            }
        }
        
        platform_times = optimal_times.get(platform, optimal_times['twitter'])
        
        # Generate recommendations for the next week
        recommendations = []
        now = datetime.now()
        
        for i in range(7):  # Next 7 days
            target_date = now + timedelta(days=i)
            is_weekend = target_date.weekday() >= 5
            
            times = platform_times['weekends'] if is_weekend else platform_times['weekdays']
            
            for time_str in times:
                hour, minute = map(int, time_str.split(':'))
                target_datetime = target_date.replace(hour=hour, minute=minute, second=0, microsecond=0)
                
                if target_datetime > now:  # Only future times
                    recommendations.append({
                        'datetime': target_datetime.isoformat(),
                        'day_type': 'weekend' if is_weekend else 'weekday',
                        'confidence': 0.8 if not is_weekend else 0.6
                    })
        
        # Sort by confidence and take top 10
        recommendations.sort(key=lambda x: x['confidence'], reverse=True)
        
        result = {
            'platform': platform,
            'timezone': timezone,
            'content_preview': content[:50] + '...' if len(content) > 50 else content,
            'optimal_times': recommendations[:10],
            'general_advice': f"For {platform}, post during {', '.join(platform_times['weekdays'])} on weekdays"
        }
        
        return json.dumps(result, indent=2)
    
    except Exception as e:
        return json.dumps({'error': str(e)})

# Create FunctionTool objects for LlamaIndex
virality_tool = FunctionTool.from_defaults(fn=analyze_content_virality)
collector_tool = FunctionTool.from_defaults(fn=collect_trending_content)
variations_tool = FunctionTool.from_defaults(fn=generate_content_variations)
search_tool = FunctionTool.from_defaults(fn=search_viral_patterns)
timing_tool = FunctionTool.from_defaults(fn=optimize_posting_time)

print("LlamaIndex tools created successfully!")

## Viral Content Predictor Agent

Now let's create the main LlamaIndex agent:

In [ ]:
class ViralContentPredictorAgent:
    """Main LlamaIndex agent for viral content prediction"""
    
    def __init__(self):
        # Initialize the ReAct agent with tools
        self.agent = ReActAgent.from_tools(
            tools=[
                virality_tool,
                collector_tool, 
                variations_tool,
                search_tool,
                timing_tool
            ],
            llm=Settings.llm,
            verbose=True
        )
        
        self.conversation_history = []
    
    def predict_virality(self, content: str, platform: str = "twitter") -> str:
        """Predict virality for given content"""
        prompt = f"""
        Analyze the viral potential of this content for {platform}:
        
        "{content}"
        
        Please provide:
        1. Virality score and confidence
        2. Predicted engagement metrics
        3. Specific recommendations for improvement
        4. Optimal hashtags for this platform
        5. Best posting time suggestions
        
        Use the available tools to analyze this content comprehensively.
        """
        
        response = self.agent.chat(prompt)
        self.conversation_history.append({"user": prompt, "agent": str(response)})
        return str(response)
    
    def analyze_trending_content(self, platform: str = "hackernews", limit: int = 20) -> str:
        """Analyze trending content from a platform"""
        prompt = f"""
        Collect and analyze trending content from {platform}. Get {limit} posts and provide:
        
        1. Overall patterns in viral content
        2. Common themes and topics
        3. Engagement metrics analysis
        4. Key insights for content creators
        5. Platform-specific recommendations
        
        Use the tools to collect data and analyze patterns.
        """
        
        response = self.agent.chat(prompt)
        self.conversation_history.append({"user": prompt, "agent": str(response)})
        return str(response)
    
    def optimize_content_for_platforms(self, content: str, platforms: List[str]) -> str:
        """Optimize content for multiple platforms"""
        platforms_str = ",".join(platforms)
        
        prompt = f"""
        Take this original content and optimize it for {platforms_str}:
        
        Original content: "{content}"
        
        For each platform, provide:
        1. Optimized content version
        2. Virality prediction score
        3. Platform-specific recommendations
        4. Optimal posting times
        5. Suggested hashtags and mentions
        
        Compare the variations and recommend the best approach for each platform.
        """
        
        response = self.agent.chat(prompt)
        self.conversation_history.append({"user": prompt, "agent": str(response)})
        return str(response)
    
    def find_viral_inspiration(self, topic: str) -> str:
        """Find viral content inspiration for a topic"""
        prompt = f"""
        Help me find viral content inspiration for the topic: "{topic}"
        
        Please:
        1. Search for similar viral content in our database
        2. Identify patterns and themes that work
        3. Suggest content angles and approaches
        4. Provide specific examples of viral posts
        5. Recommend hashtags and timing strategies
        
        Use the search tools to find relevant viral patterns.
        """
        
        response = self.agent.chat(prompt)
        self.conversation_history.append({"user": prompt, "agent": str(response)})
        return str(response)
    
    def comprehensive_content_strategy(self, business_goal: str, target_audience: str) -> str:
        """Create a comprehensive viral content strategy"""
        prompt = f"""
        Create a comprehensive viral content strategy with these parameters:
        
        Business Goal: {business_goal}
        Target Audience: {target_audience}
        
        Please provide:
        1. Content themes and topics that would resonate
        2. Platform-specific strategies
        3. Optimal posting schedule and timing
        4. Hashtag and engagement strategies
        5. Content variations for A/B testing
        6. Success metrics to track
        
        Use all available tools to research trending patterns and create data-driven recommendations.
        """
        
        response = self.agent.chat(prompt)
        self.conversation_history.append({"user": prompt, "agent": str(response)})
        return str(response)
    
    def get_conversation_history(self) -> List[Dict[str, str]]:
        """Get the conversation history"""
        return self.conversation_history
    
    def reset_conversation(self):
        """Reset the conversation history"""
        self.conversation_history = []
        self.agent.reset()

# Initialize the agent
viral_agent = ViralContentPredictorAgent()
print("Viral Content Predictor Agent initialized!")

## Demo: Viral Content Prediction

Let's demonstrate the viral content predictor:

In [ ]:
# Test individual tools first
print("=== Testing Individual Tools ===")

# Test content analysis
sample_content = "🚀 Just discovered an AI tool that writes code 10x faster than humans! This is going to change everything in software development. What do you think? #AI #Programming #Tech"

print("\n1. Analyzing content virality:")
print("Content:", sample_content)
result = analyze_content_virality(sample_content, "twitter")
print("\nVirality Analysis:")
print(result)

# Test content variations
print("\n2. Generating content variations:")
variations = generate_content_variations(sample_content, "twitter,linkedin,tiktok")
print("\nContent Variations:")
print(variations)

# Test timing optimization
print("\n3. Optimizing posting time:")
timing = optimize_posting_time(sample_content, "twitter")
print("\nOptimal Timing:")
print(timing)

In [ ]:
# Test trending content collection
print("=== Testing Content Collection ===")

# Collect mock data for testing
print("\nCollecting mock trending content:")
trending_result = collect_trending_content("mock_data", 5)
print(trending_result)

# Try collecting from Hacker News (real data)
print("\n\nCollecting from Hacker News:")
hn_result = collect_trending_content("hackernews", 10)
print(hn_result[:500] + "..." if len(hn_result) > 500 else hn_result)

In [ ]:
# Test the LlamaIndex agent
print("=== Testing LlamaIndex Agent ===")

# Test 1: Simple virality prediction
test_content = "Breaking: Scientists discover new planet that could support life! 🌍✨ This changes everything we know about space exploration. #Space #Science #Discovery"

print("\nTest 1: Virality Prediction")
print(f"Content: {test_content}")
print("\nAgent Response:")

try:
    response = viral_agent.predict_virality(test_content, "twitter")
    print(response)
except Exception as e:
    print(f"Note: Agent requires OpenAI API. Simulating response...")
    # Simulate agent response
    mock_response = f"""
Based on my analysis using the virality prediction tools:

**Virality Score: 78/100** (High potential)
**Confidence: 75%**

**Predicted Engagement:**
- Likes: ~1,560
- Shares: ~312
- Comments: ~234
- Views: ~15,600

**Key Factors Contributing to High Score:**
- Strong emotional trigger words ("Breaking", "discover")
- Science topic has broad appeal
- Optimal use of emojis and hashtags
- Creates sense of urgency and importance

**Recommendations:**
1. Post during peak hours (12-3 PM or 6-9 PM)
2. Consider adding a call-to-action ("What do you think?")
3. Could benefit from a link to credible source

**Suggested Hashtags:** #Space #Science #Discovery #Astronomy #Breaking
"""
    print(mock_response)

In [ ]:
# Test multi-platform optimization
print("\n=== Multi-Platform Optimization Test ===")

original_content = "The future of work is changing. Remote work, AI automation, and flexible schedules are becoming the norm. Companies that adapt will thrive."

print(f"Original content: {original_content}")
print("\nOptimizing for multiple platforms...")

try:
    response = viral_agent.optimize_content_for_platforms(original_content, ["twitter", "linkedin", "tiktok"])
    print(response)
except Exception as e:
    print(f"Note: Agent requires OpenAI API. Showing tool output directly...")
    
    # Use the tool directly
    direct_result = generate_content_variations(original_content, "twitter,linkedin,tiktok")
    print("\nDirect tool output:")
    print(direct_result)

In [ ]:
# Test viral pattern search
print("\n=== Viral Pattern Search Test ===")

search_query = "AI and technology trends"
print(f"Searching for viral patterns: {search_query}")

search_result = search_viral_patterns(search_query, 3)
print("\nSearch Results:")
print(search_result)

# Demonstrate comprehensive strategy
print("\n=== Comprehensive Strategy Demo ===")

business_goal = "Increase brand awareness for a SaaS productivity tool"
target_audience = "Remote workers, entrepreneurs, and small business owners aged 25-45"

print(f"Business Goal: {business_goal}")
print(f"Target Audience: {target_audience}")

# Simulate comprehensive strategy response
mock_strategy = f"""
**COMPREHENSIVE VIRAL CONTENT STRATEGY**

**Content Themes & Topics:**
1. Productivity hacks and time-saving tips
2. Remote work challenges and solutions
3. Entrepreneurship success stories
4. Behind-the-scenes of building a SaaS
5. Customer success and transformation stories

**Platform-Specific Strategies:**

*Twitter:*
- Quick tips and statistics (280 chars)
- Thread series on productivity
- Real-time engagement with community
- Hashtags: #productivity #remotework #saas

*LinkedIn:*
- Thought leadership articles
- Professional insights and case studies
- Employee and customer spotlights
- Industry trend analysis

*TikTok:*
- Day-in-the-life content
- Quick productivity demos
- Trending audio with productivity tips
- Behind-the-scenes office content

**Optimal Posting Schedule:**
- Twitter: 3x daily (9 AM, 1 PM, 6 PM EST)
- LinkedIn: 1x daily (8 AM EST weekdays)
- TikTok: 1x daily (6-8 PM EST)

**Content Variations for A/B Testing:**
1. Question vs. Statement formats
2. With vs. without emojis
3. Different hashtag combinations
4. Video vs. text-only posts

**Success Metrics:**
- Engagement rate (aim for >3%)
- Click-through rate to website
- Brand mention increase
- Lead generation from social
- Follower growth rate
"""

print(mock_strategy)

## Performance Analytics and Insights

Let's create a system for tracking and analyzing performance:

In [ ]:
class ViralContentAnalytics:
    """Analytics system for tracking viral content performance"""
    
    def __init__(self):
        self.predictions = []
        self.actual_results = []
        self.performance_data = pd.DataFrame()
    
    def log_prediction(self, prediction: ViralityPrediction):
        """Log a virality prediction"""
        self.predictions.append({
            'timestamp': datetime.now(),
            'content': prediction.content,
            'platform': prediction.platform,
            'predicted_score': prediction.virality_score,
            'predicted_engagement': prediction.engagement_prediction,
            'confidence': prediction.confidence
        })
    
    def log_actual_result(self, content: str, platform: str, actual_engagement: Dict[str, int]):
        """Log actual engagement results"""
        self.actual_results.append({
            'timestamp': datetime.now(),
            'content': content,
            'platform': platform,
            'actual_engagement': actual_engagement
        })
    
    def calculate_prediction_accuracy(self) -> Dict[str, float]:
        """Calculate prediction accuracy metrics"""
        if len(self.predictions) == 0 or len(self.actual_results) == 0:
            return {'error': 'Insufficient data for accuracy calculation'}
        
        # Match predictions with results (simplified matching by content)
        matched_pairs = []
        
        for pred in self.predictions:
            for actual in self.actual_results:
                if pred['content'] == actual['content'] and pred['platform'] == actual['platform']:
                    matched_pairs.append((pred, actual))
                    break
        
        if not matched_pairs:
            return {'error': 'No matching prediction-result pairs found'}
        
        # Calculate accuracy metrics
        score_errors = []
        engagement_errors = []
        
        for pred, actual in matched_pairs:
            # Score accuracy (normalized)
            actual_total = sum(actual['actual_engagement'].values())
            predicted_total = sum(pred['predicted_engagement'].values())
            
            if predicted_total > 0:
                error = abs(actual_total - predicted_total) / predicted_total
                engagement_errors.append(error)
            
            score_errors.append(abs(pred['predicted_score'] - 50))  # Simplified scoring
        
        return {
            'matched_predictions': len(matched_pairs),
            'average_score_accuracy': 100 - np.mean(score_errors) if score_errors else 0,
            'average_engagement_accuracy': 100 - (np.mean(engagement_errors) * 100) if engagement_errors else 0,
            'confidence_correlation': np.corrcoef(
                [p[0]['confidence'] for p in matched_pairs],
                [abs(sum(p[1]['actual_engagement'].values()) - sum(p[0]['predicted_engagement'].values())) for p in matched_pairs]
            )[0, 1] if len(matched_pairs) > 1 else 0
        }
    
    def generate_insights_report(self) -> str:
        """Generate insights report from collected data"""
        if not self.predictions:
            return "No prediction data available for analysis."
        
        # Platform performance
        platform_scores = defaultdict(list)
        content_lengths = []
        hashtag_usage = []
        
        for pred in self.predictions:
            platform_scores[pred['platform']].append(pred['predicted_score'])
            content_lengths.append(len(pred['content']))
            hashtag_usage.append(pred['content'].count('#'))
        
        # Calculate platform averages
        platform_avg = {platform: np.mean(scores) for platform, scores in platform_scores.items()}
        
        # Content analysis
        avg_length = np.mean(content_lengths)
        avg_hashtags = np.mean(hashtag_usage)
        
        report = f"""
VIRAL CONTENT INSIGHTS REPORT
Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}

PREDICTION SUMMARY:
- Total predictions made: {len(self.predictions)}
- Average virality score: {np.mean([p['predicted_score'] for p in self.predictions]):.1f}/100
- Average confidence: {np.mean([p['confidence'] for p in self.predictions]):.1f}

PLATFORM PERFORMANCE:
{''.join([f'- {platform}: {score:.1f}/100 average score\n' for platform, score in platform_avg.items()])}

CONTENT PATTERNS:
- Average content length: {avg_length:.0f} characters
- Average hashtags per post: {avg_hashtags:.1f}
- Most successful content length: {self._find_optimal_length()}

RECOMMENDATIONS:
{self._generate_recommendations()}

ACCURACY METRICS:
{json.dumps(self.calculate_prediction_accuracy(), indent=2)}
"""
        
        return report
    
    def _find_optimal_length(self) -> str:
        """Find optimal content length based on predictions"""
        if not self.predictions:
            return "Insufficient data"
        
        length_scores = [(len(p['content']), p['predicted_score']) for p in self.predictions]
        
        # Group by length ranges
        short = [score for length, score in length_scores if length < 100]
        medium = [score for length, score in length_scores if 100 <= length < 200]
        long_posts = [score for length, score in length_scores if length >= 200]
        
        ranges = {
            'Short (< 100 chars)': np.mean(short) if short else 0,
            'Medium (100-200 chars)': np.mean(medium) if medium else 0,
            'Long (200+ chars)': np.mean(long_posts) if long_posts else 0
        }
        
        optimal = max(ranges.items(), key=lambda x: x[1])
        return f"{optimal[0]} (avg score: {optimal[1]:.1f})"
    
    def _generate_recommendations(self) -> str:
        """Generate actionable recommendations"""
        recommendations = []
        
        if self.predictions:
            avg_score = np.mean([p['predicted_score'] for p in self.predictions])
            
            if avg_score < 50:
                recommendations.append("- Focus on more engaging content with emotional triggers")
            
            hashtag_usage = np.mean([p['content'].count('#') for p in self.predictions])
            if hashtag_usage < 2:
                recommendations.append("- Increase hashtag usage (aim for 2-3 per post)")
            
            recommendations.append("- Continue A/B testing different content formats")
            recommendations.append("- Monitor optimal posting times for your audience")
        
        return '\n'.join(recommendations) if recommendations else "- Keep tracking data for better insights"

# Initialize analytics
analytics = ViralContentAnalytics()

# Log some sample predictions for demonstration
sample_predictions = [
    analyzer.predict_virality("AI is revolutionizing healthcare! 🏥 #AI #Healthcare", "twitter"),
    analyzer.predict_virality("5 productivity tips that changed my life completely", "linkedin"),
    analyzer.predict_virality("This coding hack will save you hours every day! ⚡", "twitter"),
    analyzer.predict_virality("Breaking: New study reveals surprising facts about remote work", "linkedin")
]

for pred in sample_predictions:
    analytics.log_prediction(pred)

print("Analytics system initialized with sample data!")

In [ ]:
# Generate and display analytics report
print("=== Viral Content Analytics Report ===")

report = analytics.generate_insights_report()
print(report)

# Test individual predictions with detailed analysis
print("\n\n=== Detailed Prediction Analysis ===")

test_contents = [
    {
        'content': '🚀 Just launched my first SaaS product! Built it in 30 days with AI tools. Revenue: $10K in first week! Here\'s how I did it... 🧵 #entrepreneurship #AI #startup',
        'platform': 'twitter'
    },
    {
        'content': 'After 10 years in corporate, I quit to build my own company. Here are 5 lessons I learned about leadership, team building, and creating value in today\'s market.',
        'platform': 'linkedin'
    },
    {
        'content': 'POV: you discover a productivity method that literally adds 3 hours to your day ✨ #productivity #lifehack #worksmarter',
        'platform': 'tiktok'
    }
]

for i, test in enumerate(test_contents, 1):
    print(f"\n--- Test {i}: {test['platform'].title()} ---")
    print(f"Content: {test['content']}")
    
    prediction = analyzer.predict_virality(test['content'], test['platform'])
    
    print(f"\nVirality Score: {prediction.virality_score}/100")
    print(f"Confidence: {prediction.confidence:.1%}")
    print(f"Predicted Engagement: {prediction.engagement_prediction}")
    print(f"Top Recommendations: {prediction.recommendations[:2]}")
    print(f"Suggested Hashtags: {prediction.hashtag_suggestions}")
    
    # Log the prediction
    analytics.log_prediction(prediction)

print("\n\n=== Final Performance Summary ===")
final_report = analytics.generate_insights_report()
print(final_report[:800] + "..." if len(final_report) > 800 else final_report)

## Features of this LlamaIndex Implementation

### Key Differences from Haystack:

1. **Agent-Based Architecture**: Uses ReAct agents with tool-calling capabilities
2. **Vector-First Approach**: Built around vector embeddings and similarity search
3. **Query Engine Integration**: Advanced query engines for content analysis
4. **Context-Aware Responses**: Maintains conversation context across interactions
5. **Tool Orchestration**: Intelligent tool selection and chaining

### Viral Content Prediction Capabilities:

- **Advanced Content Analysis**: AST-based feature extraction with sentiment analysis
- **Platform-Specific Optimization**: Tailored predictions for different social media platforms
- **Real-Time Trend Analysis**: Collects and analyzes trending content from multiple sources
- **A/B Testing Support**: Generates content variations for optimization
- **Performance Analytics**: Tracks prediction accuracy and generates insights

### Advanced Features:

1. **Vector Search**: Find similar viral content patterns using embeddings
2. **Content Variations**: Automatic generation of platform-optimized versions
3. **Timing Optimization**: Predict optimal posting times based on platform data
4. **Engagement Prediction**: Forecast likes, shares, comments, and views
5. **Viral Pattern Recognition**: Identify successful content patterns and themes

### Real-World Applications:

1. **Social Media Marketing**: Optimize content for maximum engagement
2. **Influencer Strategy**: Data-driven content creation for influencers
3. **Brand Marketing**: Viral marketing campaigns for businesses
4. **Content Creation**: AI-assisted content optimization for creators
5. **Market Research**: Understanding viral content trends and patterns

### Technical Innovation:

- **Multi-Modal Analysis**: Text, sentiment, timing, and engagement pattern analysis
- **Predictive Modeling**: Machine learning models for virality scoring
- **Real-Time Data**: Live trending content collection and analysis
- **Cross-Platform Intelligence**: Understanding of different platform algorithms

This implementation showcases LlamaIndex's strength in creating intelligent, context-aware agents that can perform sophisticated content analysis and prediction tasks while maintaining conversation flow and learning from interactions.